In [ ]:
import openmm as mm
from openmm import app, unit
from openmm.app.gromacsgrofile import GromacsGroFile

import numpy as np
import sys

from openmmnapshift.utils import RESIDUE_TYPES, get_napshift_force

This tutorial sets up a simulation of a short helical peptide with Chemical Shift restraints applied on top of the CHARMM27 forcefield. 

Staring from an extended conformation, the CS restraints should guide the system to a helical state.

## Define simulation parameters

In [ ]:
temperature = 298*unit.kelvin
timestep = 2*unit.femtosecond
pressure = 1*unit.bar
collision_frequency = 1/unit.picosecond

max_K = 25
K_gradient = 0.001

report_interval = 1000

## Set up CHARMM27 system

In [ ]:
gro = app.GromacsGroFile(f'Data/ShortHelix/system.gro')
top = app.GromacsTopFile(f'Data/ShortHelix/system.top', periodicBoxVectors=gro.getPeriodicBoxVectors(),
        includeDir=f'Data/GromacsForceFields/top')
system = top.createSystem(nonbondedMethod=app.PME, nonbondedCutoff=1*unit.nanometer,
        constraints=app.HBonds)

system.addForce(mm.AndersenThermostat(temperature, collision_frequency))
system.addForce(mm.MonteCarloBarostat(pressure, temperature))

## Add Chemical Shift restraints

In [ ]:
napshift_force = get_napshift_force(top.topology, 'Data/ShortHelix/CS.txt', model_type='all_atom')
napshift_force.setUsesPeriodicBoundaryConditions(True)
system.addForce(napshift_force)

## Create the OpenMM simulation

In [ ]:
integrator = mm.VerletIntegrator(timestep)
platform = mm.Platform.getPlatformByName("CUDA")
simulation = app.Simulation(top.topology, system, integrator, platform, {"Precision" : "mixed", 'DeviceIndex' : "0"})
simulation.context.setPositions(gro.positions)
simulation.minimizeEnergy()
simulation.context.setVelocitiesToTemperature(temperature)

## Add reporters

In [ ]:
xtc_reporter = app.XTCReporter('Data/ShortHelix/output.xtc', report_interval, append=False, enforcePeriodicBox=True, atomSubset=[atom.index for atom in top.topology.atoms() if atom.residue.name in RESIDUE_TYPES.keys()])
state_data_reporter = app.StateDataReporter(sys.stdout, report_interval, step=True, time=True, potentialEnergy=True, kineticEnergy=True, totalEnergy=True, temperature=True, volume=True, speed=True)
simulation.reporters.append(xtc_reporter)
simulation.reporters.append(state_data_reporter)

## Simulate!

In [ ]:
warmup_steps = int(np.floor(max_K/K_gradient))
print(f"Warming up CS restraints for {len(range(warmup_steps))} steps")
for i in range(warmup_steps):
    simulation.step(1)
    simulation.context.setParameter('NapShift_K', (i*K_gradient))
    
print(f"Simulating with CS restraints")
simulation.step(100000)

The output trajectory will be written to Data/ShortHelix